# Studio Proyek 3 — Fine-Tuning Transformer Pretrained untuk NLP

**Mata Kuliah:** Pembelajaran Mendalam (IF25-40401)
**Program Studi:** Teknik Informatika, Institut Teknologi Sumatera (ITERA)
**Dataset Utama:** SmSA (IndoNLU — Wilie dkk., 2020) — klasifikasi sentimen 3 kelas (*negative*, *neutral*, *positive*)
**Batas Waktu:** 20 November 2026

Notebook ini dibangun mengikuti **Bab 7 panduan.md** (SEL 1–9). Dijalankan di **Google Colab / Kaggle (GPU T4 gratis)**.
Pada mode Colab, seluruh artefak (dataset, log CSV, grafik, 1 model terbaik) tersimpan otomatis ke
`MyDrive/Studio3_NLP/` sehingga tidak hilang saat sesi terputus.

## Sel Rancangan (Soal 1 — Rancang Dulu Sebelum Ngoding)

Sel ini wajib terisi **sebelum eksperimen pertama dijalankan** (Bab 4 panduan.md).

### Matriks Eksperimen (3 Backbone x 3 Strategi x 3 Seed = 27 Run)

| ID | Backbone | Strategi Fine-Tuning | Variasi Random Seed |
|:---:|:---|:---|:---:|
| **E1** | `indobenchmark/indobert-base-p1` | Full Fine-Tuning | 42, 123, 2024 |
| **E2** | `indobenchmark/indobert-base-p1` | Partial Freezing (Freeze Emb + 6 Layer) | 42, 123, 2024 |
| **E3** | `indobenchmark/indobert-base-p1` | Linear Probing (Freeze 100% Backbone) | 42, 123, 2024 |
| **E4** | `bert-base-multilingual-cased` | Full Fine-Tuning | 42, 123, 2024 |
| **E5** | `bert-base-multilingual-cased` | Partial Freezing (Freeze Emb + 6 Layer) | 42, 123, 2024 |
| **E6** | `bert-base-multilingual-cased` | Linear Probing (Freeze 100% Backbone) | 42, 123, 2024 |
| **E7** | `distilbert-base-multilingual-cased` | Full Fine-Tuning | 42, 123, 2024 |
| **E8** | `distilbert-base-multilingual-cased` | Partial Freezing (Freeze Emb + 3 Layer) | 42, 123, 2024 |
| **E9** | `distilbert-base-multilingual-cased` | Linear Probing (Freeze 100% Backbone) | 42, 123, 2024 |

### Variabel Terkendali yang Dikunci
| Variabel | Nilai | Catatan |
|:---|:---|:---|
| `max_length` | 128 token | mengikuti Bab 4.2 panduan.md |
| `batch_size` | 32 | sama untuk semua run |
| `num_epochs` | 4 | tetap, tanpa early stopping |
| Optimizer | AdamW (`weight_decay = 0.01`) | beta default PyTorch |
| LR schedule | linear warmup ratio 0.1 → linear decay | per strategi: full/partial 3e-5, probing 1e-3 |
| Gradient clipping | `max_grad_norm = 1.0` | — |
| Presisi | FP16 (mixed precision, AMP) | efisiensi GPU T4 |
| Seed | 42, 123, 2024 | sama untuk semua kombinasi |
| Himpunan evaluasi | `valid_preprocess.tsv` (pembagian resmi) | pelaporan Mean ± Std antar-seed |

### Alasan Desain Singkat
- **Dataset SmSA dipilih** karena merupakan benchmark resmi klasifikasi sentimen Bahasa Indonesia
  (3 kelas) dan menjadi bagian dari IndoNLU, sehingga perbandingan backbone berbahasa Indonesia vs
  multibahasa dapat dilakukan secara adil pada domain yang sama.
- **Ukuran subset:** seluruh data latih resmi (tanpa subset) agar hasil tidak bergantung pada cuplikan acak.
- **3 seed per kombinasi:** untuk mengukur variasi acak antar-seed, sehingga klaim keunggulan sebuah
  backbone dapat dibandingkan terhadap besaran simpangan bakunya (bukan hanya satu kali coba).

### Hipotesis Penelitian
1. **H1 (Kesesuaian Bahasa Korpus):** IndoBERT mencapai F1 Macro dan Akurasi tertinggi secara
   signifikan dibandingkan mBERT dan DistilBERT pada mode Full Fine-Tuning, dengan selisih performa
   yang melampaui variasi acak antar-seed.
2. **H2 (Efisiensi vs Penurunan Performa DistilBERT):** DistilBERT Multilingual mempercepat waktu
   pelatihan dan latensi inferensi per kalimat >= 35% dibanding mBERT, dengan penurunan F1 Macro
   tidak melebihi 3.0 poin persentase.
3. **H3 (Pengaruh Pembekuan Lapisan):** F1 Macro mengikuti hierarki
   Full Fine-Tuning >= Partial Freezing > Linear Probing; partial freezing mempertahankan >= 96%
   performa full fine-tuning dengan penghematan waktu latih dan memori GPU >= 40%.

### Catatan Distribusi Kelas dan Metrik Utama
Distribusi kelas SmSA tidak seimbang (positive ~58%, negative ~31%, neutral ~11%). Karena kelas
`neutral` merupakan minoritas, **Macro F1-Score** dijadikan metrik utama penentu keunggulan model
(akurasi saja dapat memberi ilusi performa tinggi).

### Struktur Folder Rujukan

```text
studio3_Deep-Learning_NLP/            <-- lokal (laptop)
├── notebooks/Studio_Proyek_3_NLP.ipynb
├── data/{train_preprocess.tsv, valid_preprocess.tsv}
├── hasil_eksperimen/{hasil_smsa_<kode>.csv, rekap_agregat_smsa_<kode>.csv, grafik_*.png}
└── laporan_dan_presentasi/

MyDrive/Studio3_NLP/                  <-- Google Drive (dibuat otomatis oleh notebook)
├── data/
└── output/{hasil_smsa_<kode>.csv, rekap_agregat_smsa_<kode>.csv, grafik_*.png, model_terbaik/}
```

## SEL 1 — Instalasi Dependensi

In [ ]:
# ==============================================================================
# SEL 1: INSTALASI DEPENDENSI (Colab sudah memuat sebagian besar; baris ini aman dijalankan ulang)
# ==============================================================================
# !pip install -q -U transformers scikit-learn seaborn matplotlib scipy

## SEL 2 — Setup Lingkungan, Google Drive Mount, & Path Otomatis

In [ ]:
# ==============================================================================
# SEL 2: SETUP LINGKUNGAN, GOOGLE DRIVE MOUNT, & PATH OTOMATIS
# ==============================================================================
import os
import gc
import time
import math
import random
from pathlib import Path
import urllib.request

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

# 1. Deteksi Lingkungan (Google Colab vs Lokal)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = Path('/content/drive/MyDrive/Studio3_NLP')
    IN_COLAB = True
    print("Mode: Google Colab (Google Drive Terhubung)")
except ImportError:
    # Fallback lokal: notebook ini berada di <repo>/notebooks/, jadi akar proyek = satu level di atasnya
    BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    IN_COLAB = False
    print(f"Mode: Lokal / Offline (BASE_DIR = {BASE_DIR})")

# 2. Definisikan Direktori Kerja
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Direktori Data   : {DATA_DIR}")
print(f"Direktori Output : {OUTPUT_DIR}")

# 3. Deteksi Akselerasi GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Perangkat Komputasi: {DEVICE}")
if torch.cuda.is_available():
    print(f"Model GPU: {torch.cuda.get_device_name(0)}")

## SEL 3 — Fungsi Seeding & Konfigurasi Hiperparameter Terkunci

In [ ]:
# ==============================================================================
# SEL 3: SEEDING, KONFIGURASI TERKUNCI, MODE EKSEKUSI & GERBANG RANCANGAN
# ==============================================================================
import json, hashlib

def set_seed(seed: int = 42):
    """Menjamin reproduksibilitas eksperimen lintas library."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

# MODE = "smoke" -> uji jalan cepat (subset kecil, 1 seed, 1 epoch), keluaran dipisah folder.
# MODE = "full"  -> matriks penuh 27 run. Mode ini menolak berjalan bila Rancangan kosong.
MODE = "full"

CONFIG = {
    "dataset_name": "SmSA_IndoNLU",
    "num_classes": 3,
    "label_map": {"negative": 0, "neutral": 1, "positive": 2},
    "inv_label_map": {0: "negative", 1: "neutral", 2: "positive"},
    "max_length": 128,
    "batch_size": 32,
    "num_epochs": 4,
    "lr_full": 3e-5,
    "lr_partial": 3e-5,
    "lr_probing": 1e-3,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "max_grad_norm": 1.0,
    "seeds": [42, 123, 2024],
    "backbones": {
        "indobert": "indobenchmark/indobert-base-p1",
        "mbert": "bert-base-multilingual-cased",
        "distilmbert": "distilbert-base-multilingual-cased"
    },
    "strategies": ["full", "partial", "probing"],
    # jumlah lapis yang dibekukan pada partial freezing (separuh lapis awal) + embedding
    "partial_frozen_layers": {"indobert": 6, "mbert": 6, "distilmbert": 3},
    "latency_samples": 300,   # kalimat valid pertama (tetap) untuk benchmark latensi
    "latency_warmup": 10,
}

if MODE == "smoke":
    CONFIG.update({"seeds": [42], "num_epochs": 1, "smoke_train_n": 256, "smoke_valid_n": 128,
                   "latency_samples": 40, "latency_warmup": 5})

# Kode konfigurasi: berubah otomatis bila konfigurasi berubah -> hasil antar-konfigurasi tidak tercampur
_hash_input = {k: CONFIG[k] for k in sorted(CONFIG) if k not in ("inv_label_map",)}
KODE = hashlib.sha1(json.dumps(_hash_input, sort_keys=True).encode()).hexdigest()[:6]
print(f"MODE = {MODE} | kode konfigurasi = {KODE}")

# ---- Gerbang Rancangan (Soal 1): ringkasan terstruktur dari sel Rancangan di atas ----
RANCANGAN = {
    "matriks": "3 backbone (IndoBERT, mBERT, DistilmBERT) x 3 strategi (full, partial, probing) x 3 seed = 27 run",
    "H1": "IndoBERT unggul pada F1 makro dibanding mBERT dan DistilmBERT (full FT), melampaui std antar-seed.",
    "H2": "DistilmBERT >=35% lebih cepat (latih & latensi) dibanding mBERT dengan penurunan F1 makro <=3 poin.",
    "H3": "F1: full >= partial > probing; partial >=96% F1 full dengan hemat waktu & memori >=40%.",
    "alasan_desain": "SmSA: benchmark sentimen Indonesia 3 kelas; seluruh data latih resmi dipakai; 3 seed untuk std.",
    "variabel_terkunci": {k: CONFIG[k] for k in ("max_length", "batch_size", "num_epochs", "seeds")},
}
if MODE == "full":
    kosong = [k for k, v in RANCANGAN.items() if not v]
    assert not kosong, f"Mode penuh DITOLAK: bagian Rancangan masih kosong -> {kosong}"
    assert len(CONFIG["seeds"]) >= 3 and len(CONFIG["backbones"]) >= 3, "Minimal 3 seed & 3 backbone."
    print("Gerbang Rancangan lolos.")

## SEL 4 — Pengunduhan Data (Persisten di Google Drive) & Pembersihan

In [ ]:
# ==============================================================================
# SEL 4: PENGUNDUHAN DATA (PERSISTEN) & PEMBERSIHAN
# ==============================================================================
if MODE == "smoke":
    OUTPUT_DIR = OUTPUT_DIR / "smoke_test"
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_URL = "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
VALID_URL = "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv"

train_file = DATA_DIR / "train_preprocess.tsv"
valid_file = DATA_DIR / "valid_preprocess.tsv"

for url, f in ((TRAIN_URL, train_file), (VALID_URL, valid_file)):
    if not f.exists():
        print(f"Mengunduh {f.name} ...")
        urllib.request.urlretrieve(url, f)
print("Dataset SmSA siap.")

df_train_raw = pd.read_csv(train_file, sep='\t', names=['text', 'label'], header=None)
df_valid_raw = pd.read_csv(valid_file, sep='\t', names=['text', 'label'], header=None)

# Urutan pembersihan PENTING: deteksi konflik label HARUS sebelum deduplikasi,
# karena setelah drop_duplicates tiap teks hanya punya satu label (konflik jadi tak terdeteksi).
n_mentah = len(df_train_raw)

# 1. Teks berlabel konfliktual (teks sama, label berbeda) -> dibuang seluruhnya
n_label = df_train_raw.groupby('text')['label'].nunique()
teks_konflik = set(n_label[n_label > 1].index)
df_train = df_train_raw[~df_train_raw['text'].isin(teks_konflik)]
n_konflik_baris = n_mentah - len(df_train)

# 2. Duplikat teks persis (label sama) -> sisakan satu
sebelum = len(df_train)
df_train = df_train.drop_duplicates(subset=['text'])
n_duplikat = sebelum - len(df_train)

# 3. Kebocoran: teks latih yang juga muncul di validasi
teks_valid = set(df_valid_raw['text'])
sebelum = len(df_train)
df_train = df_train[~df_train['text'].isin(teks_valid)].copy()
n_bocor = sebelum - len(df_train)

df_train['label_id'] = df_train['label'].map(CONFIG['label_map'])
df_valid = df_valid_raw.copy()
df_valid['label_id'] = df_valid['label'].map(CONFIG['label_map'])
assert df_train['label_id'].notna().all() and df_valid['label_id'].notna().all(), "Ada label di luar label_map!"
df_train['label_id'] = df_train['label_id'].astype(int)
df_valid['label_id'] = df_valid['label_id'].astype(int)

print(f"Mentah={n_mentah} | baris konflik dibuang={n_konflik_baris} ({len(teks_konflik)} teks) | "
      f"duplikat dibuang={n_duplikat} | bocor ke validasi dibuang={n_bocor}")
print(f"Data Bersih: Train={len(df_train)}, Valid={len(df_valid)}")
print("Distribusi Kelas Data Latih (%):")
print((df_train['label'].value_counts(normalize=True) * 100).round(2))

if MODE == "smoke":
    df_train = df_train.sample(CONFIG["smoke_train_n"], random_state=0).reset_index(drop=True)
    df_valid = df_valid.sample(CONFIG["smoke_valid_n"], random_state=0).reset_index(drop=True)
    print(f"[SMOKE] subset: train={len(df_train)}, valid={len(df_valid)}")

## SEL 5 — PyTorch Dataset & Layer Freezing Logic

In [ ]:
# ==============================================================================
# SEL 5: DATASET (PRA-TOKENISASI + PADDING DINAMIS) & LOGIKA FREEZING
# ==============================================================================
class SmSADataset(Dataset):
    """Tokenisasi dilakukan SEKALI di awal (tanpa padding); padding dilakukan per-batch oleh collator
    sehingga tiap batch hanya sepanjang kalimat terpanjangnya (maks. max_length) -> jauh lebih efisien."""
    def __init__(self, texts, labels, tokenizer, max_len=128):
        enc = tokenizer(list(map(str, texts)), truncation=True, max_length=max_len,
                        padding=False, return_attention_mask=True)
        self.input_ids = enc['input_ids']
        self.attention_mask = enc['attention_mask']
        self.labels = list(labels)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {'input_ids': self.input_ids[idx],
                'attention_mask': self.attention_mask[idx],
                'label': self.labels[idx]}

def buat_collator(tokenizer):
    def collate(batch):
        labels = torch.tensor([b['label'] for b in batch], dtype=torch.long)
        padded = tokenizer.pad(
            [{'input_ids': b['input_ids'], 'attention_mask': b['attention_mask']} for b in batch],
            padding=True, return_tensors='pt')
        return {'input_ids': padded['input_ids'],
                'attention_mask': padded['attention_mask'],
                'label': labels}
    return collate

def hitung_parameter(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def _komponen_backbone(model):
    """Mengembalikan (embeddings, daftar_layer_encoder, modul_head) untuk arsitektur BERT / DistilBERT."""
    if hasattr(model, 'bert'):
        return model.bert.embeddings, list(model.bert.encoder.layer), [model.classifier]
    if hasattr(model, 'distilbert'):
        # DistilBERT punya head 2 lapis: pre_classifier + classifier -> keduanya bagian head
        return model.distilbert.embeddings, list(model.distilbert.transformer.layer), [model.pre_classifier, model.classifier]
    raise ValueError("Arsitektur tidak dikenali (hanya BERT / DistilBERT).")

def atur_strategi_fine_tuning(model, backbone_key: str, strategi: str):
    """Mengatur requires_grad sesuai strategi, lalu MEMVERIFIKASI hasilnya (bukan sekadar mengasumsikan)."""
    emb, layers, head = _komponen_backbone(model)
    n_layer = len(layers)

    for p in model.parameters():          # titik awal seragam
        p.requires_grad = True

    if strategi == "full":
        pass

    elif strategi == "partial":
        n_beku = CONFIG['partial_frozen_layers'][backbone_key]
        assert n_beku == n_layer // 2, f"{backbone_key}: {n_beku} bukan separuh dari {n_layer} lapis"
        for p in emb.parameters():
            p.requires_grad = False
        for layer in layers[:n_beku]:
            for p in layer.parameters():
                p.requires_grad = False

    elif strategi == "probing":
        for p in model.parameters():
            p.requires_grad = False
        for m in head:
            for p in m.parameters():
                p.requires_grad = True
    else:
        raise ValueError(strategi)

    # ---- Verifikasi eksplisit ----
    if strategi == "partial":
        beku = all(not p.requires_grad for p in emb.parameters()) and \
               all(not p.requires_grad for l in layers[:n_beku] for p in l.parameters())
        aktif = all(p.requires_grad for l in layers[n_beku:] for p in l.parameters()) and \
                all(p.requires_grad for m in head for p in m.parameters())
        assert beku and aktif, "Freezing partial tidak sesuai definisi."
    if strategi == "probing":
        id_head = {id(p) for m in head for p in m.parameters()}
        assert all(p.requires_grad == (id(p) in id_head) for p in model.parameters()), \
            "Probing: hanya head yang boleh dilatih."

    tot, train = hitung_parameter(model)
    print(f"[{backbone_key.upper()} - {strategi.upper()}] Param Total: {tot:,} | Trainable: {train:,} ({train/tot*100:.2f}%) "
          f"| lapis={n_layer} | head={[type(m).__name__ for m in head]}")
    return model

## SEL 6 — Training, Benchmark Inferensi, & Evaluasi

In [ ]:
# ==============================================================================
# SEL 6: TRAINING, BENCHMARK INFERENSI, & EVALUASI
# ==============================================================================
USE_AMP = torch.cuda.is_available()   # FP16 hanya bila ada GPU (CPU: FP32)

def ukur_latensi_inferensi(model, dataset, device, n_sample, n_warmup):
    """Latensi per KALIMAT (batch=1, tanpa padding, FP32), memakai n_sample kalimat valid pertama yang SAMA
    untuk semua model. Warmup tidak dihitung; setiap pengukuran dibatasi cuda.synchronize()."""
    model.eval()
    cuda = device.type == "cuda"
    waktu = []
    n_total = min(n_sample + n_warmup, len(dataset))
    with torch.inference_mode():
        for i in range(n_total):
            ids = torch.tensor([dataset.input_ids[i]], device=device)
            msk = torch.tensor([dataset.attention_mask[i]], device=device)
            if cuda: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(input_ids=ids, attention_mask=msk)
            if cuda: torch.cuda.synchronize()
            t1 = time.perf_counter()
            if i >= n_warmup:
                waktu.append(t1 - t0)
    return float(np.mean(waktu) * 1000.0)

def latih_satu_run(model, train_loader, val_loader, val_dataset, optimizer, scheduler, epochs, device):
    criterion = nn.CrossEntropyLoss()
    cuda = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    if cuda:
        torch.cuda.synchronize()
        base_mb = torch.cuda.memory_allocated(device) / 1024**2
        if base_mb > 50:
            print(f"[PERINGATAN] Memori GPU sebelum run sudah {base_mb:.0f} MB (kemungkinan sisa run sebelumnya).")
    model.to(device)
    if cuda:
        torch.cuda.reset_peak_memory_stats(device)   # puncak dihitung sejak bobot model berada di GPU
        torch.cuda.synchronize()

    t_mulai = time.perf_counter()
    for epoch in range(epochs):
        model.train()
        for batch in train_loader:
            optimizer.zero_grad(set_to_none=True)
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits.float(), labels)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG['max_grad_norm'])
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
    if cuda: torch.cuda.synchronize()
    waktu_latih_detik = time.perf_counter() - t_mulai
    memori_puncak_mb = torch.cuda.max_memory_allocated(device) / 1024**2 if cuda else 0.0

    # Evaluasi pada himpunan validasi resmi
    model.eval()
    semua_pred, semua_label = [], []
    with torch.inference_mode():
        for batch in val_loader:
            logits = model(input_ids=batch['input_ids'].to(device),
                           attention_mask=batch['attention_mask'].to(device)).logits
            semua_pred.extend(torch.argmax(logits, dim=1).cpu().tolist())
            semua_label.extend(batch['label'].tolist())

    f1_kelas = f1_score(semua_label, semua_pred, average=None, labels=[0, 1, 2])
    latensi_ms = ukur_latensi_inferensi(model, val_dataset, device,
                                        CONFIG['latency_samples'], CONFIG['latency_warmup'])
    return {
        "accuracy": accuracy_score(semua_label, semua_pred),
        "f1_macro": f1_score(semua_label, semua_pred, average='macro'),
        "f1_negative": f1_kelas[0], "f1_neutral": f1_kelas[1], "f1_positive": f1_kelas[2],
        "train_time_s": waktu_latih_detik,
        "latency_ms": latensi_ms,
        "peak_memory_mb": memori_puncak_mb,
    }

## SEL 7 — Eksekusi Matriks Lengkap & Auto-Save ke Google Drive

In [ ]:
# ==============================================================================
# SEL 7: EKSEKUSI MATRIKS LENGKAP & AUTO-SAVE (ATOMIK) KE GOOGLE DRIVE
# ==============================================================================
berkas_csv_hasil = OUTPUT_DIR / f"hasil_smsa_{KODE}.csv"
best_model_path = OUTPUT_DIR / "model_terbaik"

def simpan_csv_atomik(df, tujuan: Path):
    """Tulis ke berkas sementara lalu ganti -> CSV tidak rusak bila sesi Colab putus saat menulis."""
    tmp = tujuan.with_suffix(".tmp")
    df.to_csv(tmp, index=False)
    os.replace(tmp, tujuan)

if berkas_csv_hasil.exists():
    hasil_semua_run = pd.read_csv(berkas_csv_hasil).to_dict('records')
    print(f"Melanjutkan eksperimen. {len(hasil_semua_run)} run sudah selesai (kode {KODE}).")
else:
    hasil_semua_run = []

# Skor terbaik diwarisi dari riwayat -> run lanjutan tidak menimpa model terbaik dengan yang lebih buruk
best_f1_global = max((r['f1_macro'] for r in hasil_semua_run), default=-1.0)
if best_f1_global > 0 and not best_model_path.exists():
    best_f1_global = -1.0   # model terbaik lama hilang -> simpan ulang pada run berikutnya

for bb_key, bb_path in CONFIG["backbones"].items():
    print(f"\n{'='*65}\nMEMPROSES BACKBONE: {bb_key.upper()} ({bb_path})\n{'='*65}")
    tokenizer = AutoTokenizer.from_pretrained(bb_path)
    collate = buat_collator(tokenizer)

    train_dataset = SmSADataset(df_train['text'], df_train['label_id'], tokenizer, CONFIG['max_length'])
    val_dataset = SmSADataset(df_valid['text'], df_valid['label_id'], tokenizer, CONFIG['max_length'])
    val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'], shuffle=False, collate_fn=collate)

    for strat in CONFIG["strategies"]:
        for seed in CONFIG["seeds"]:
            if any(r['backbone'] == bb_key and r['strategy'] == strat and r['seed'] == seed
                   for r in hasil_semua_run):
                print(f"[SKIP] {bb_key} | {strat} | seed {seed} sudah tercatat.")
                continue

            print(f"\n---> Running: Backbone={bb_key} | Strategi={strat} | Seed={seed}")
            set_seed(seed)
            g = torch.Generator().manual_seed(seed)   # urutan shuffle ditentukan seed, bukan RNG global
            train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'], shuffle=True,
                                      collate_fn=collate, generator=g)

            # Head Replacement: head klasifikasi baru (num_labels=3) diinisialisasi acak setelah set_seed
            model = AutoModelForSequenceClassification.from_pretrained(
                bb_path, num_labels=CONFIG['num_classes'])
            assert model.classifier.out_features == CONFIG['num_classes']
            model = atur_strategi_fine_tuning(model, bb_key, strat)
            total_param, trainable_param = hitung_parameter(model)

            lr = {"full": CONFIG['lr_full'], "partial": CONFIG['lr_partial'],
                  "probing": CONFIG['lr_probing']}[strat]
            optimizer = torch.optim.AdamW(
                [p for p in model.parameters() if p.requires_grad],
                lr=lr, weight_decay=CONFIG['weight_decay'])
            total_steps = len(train_loader) * CONFIG['num_epochs']
            scheduler = get_linear_schedule_with_warmup(
                optimizer, num_warmup_steps=int(total_steps * CONFIG['warmup_ratio']),
                num_training_steps=total_steps)

            res = latih_satu_run(model, train_loader, val_loader, val_dataset,
                                 optimizer, scheduler, CONFIG['num_epochs'], DEVICE)

            catatan = {"backbone": bb_key, "strategy": strat, "seed": seed, "lr": lr, **res,
                       "total_params": total_param, "trainable_params": trainable_param}
            hasil_semua_run.append(catatan)
            simpan_csv_atomik(pd.DataFrame(hasil_semua_run), berkas_csv_hasil)
            print(f"Tersimpan -> {berkas_csv_hasil.name} | F1={res['f1_macro']:.4f} Acc={res['accuracy']:.4f} "
                  f"Latih={res['train_time_s']:.0f}s Lat={res['latency_ms']:.1f}ms Mem={res['peak_memory_mb']:.0f}MB")

            if res["f1_macro"] > best_f1_global:
                best_f1_global = res["f1_macro"]
                model.save_pretrained(best_model_path)
                tokenizer.save_pretrained(best_model_path)
                print(f"Model terbaik baru ({bb_key}-{strat}-seed{seed}) disimpan.")

            # Pembersihan memori GPU antar-run
            del model, optimizer, scheduler, train_loader, res
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    del train_dataset, val_dataset, val_loader
    gc.collect()

print("\nSeluruh eksperimen selesai.")

## SEL 8 — Rekapitulasi Agregat (Mean + Std)

In [ ]:
# ==============================================================================
# SEL 8: REKAPITULASI AGREGAT (MEAN ± STD, ddof=1) + VALIDASI KELENGKAPAN
# ==============================================================================
df_hasil = pd.DataFrame(hasil_semua_run)

# Validasi kelengkapan matriks & hierarki parameter yang dilatih (probing < partial < full)
n_harapan = len(CONFIG["backbones"]) * len(CONFIG["strategies"]) * len(CONFIG["seeds"])
assert len(df_hasil) == n_harapan, f"Run belum lengkap: {len(df_hasil)}/{n_harapan}"
for bb, g in df_hasil.groupby("backbone"):
    tp = g.groupby("strategy")["trainable_params"].first()
    assert tp["probing"] < tp["partial"] < tp["full"], f"Hierarki parameter salah pada {bb}: {tp.to_dict()}"
print("Validasi lolos: matriks lengkap & parameter dilatih probing < partial < full.")

metrik = ["accuracy", "f1_macro", "f1_neutral", "train_time_s", "latency_ms", "peak_memory_mb"]
rekap_agregat = df_hasil.groupby(["backbone", "strategy"]).agg(
    n_seed=("seed", "nunique"),
    total_params=("total_params", "first"),
    trainable_params=("trainable_params", "first"),
    **{f"{m}_{s}": (m, s) for m in metrik for s in ("mean", "std")}
).reset_index()
for m in metrik:   # kolom tampilan "mean ± std"
    rekap_agregat[f"{m}_ringkas"] = rekap_agregat.apply(
        lambda r: f"{r[m+'_mean']:.4f} ± {r[m+'_std']:.4f}" if m.startswith(("acc", "f1")) else
                  f"{r[m+'_mean']:.2f} ± {r[m+'_std']:.2f}", axis=1)

berkas_rekap = OUTPUT_DIR / f"rekap_agregat_smsa_{KODE}.csv"
simpan_csv_atomik(rekap_agregat, berkas_rekap)
print(f"Rekap agregat tersimpan di {berkas_rekap}")
display(rekap_agregat[["backbone", "strategy", "n_seed", "trainable_params"] + [f"{m}_ringkas" for m in metrik]])

## SEL 9 — Visualisasi Otomatis & Penyimpanan Grafik

In [ ]:
# ==============================================================================
# SEL 9: VISUALISASI OTOMATIS & PENYIMPANAN GRAFIK
# ==============================================================================
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Barplot F1 Macro
sns.barplot(data=df_hasil, x="backbone", y="f1_macro", hue="strategy", errorbar="sd", ax=axes[0, 0])
axes[0, 0].set_title("Perbandingan F1 Macro (Mean ± Std antar-seed)", fontsize=13, fontweight='bold')
axes[0, 0].set_ylim(0.5, 1.0)

# 2. Barplot Waktu Pelatihan
sns.barplot(data=df_hasil, x="backbone", y="train_time_s", hue="strategy", errorbar="sd", ax=axes[0, 1])
axes[0, 1].set_title("Waktu Pelatihan per Run (Detik)", fontsize=13, fontweight='bold')

# 3. Trade-off F1 vs Latensi Inferensi
sns.scatterplot(
    data=df_hasil, x="latency_ms", y="f1_macro", hue="backbone", style="strategy",
    s=150, ax=axes[1, 0]
)
axes[1, 0].set_title("Trade-off: F1 Macro vs Latensi Inferensi (ms/kalimat)", fontsize=13, fontweight='bold')

# 4. Konsumsi Memori GPU Puncak
sns.barplot(data=df_hasil, x="backbone", y="peak_memory_mb", hue="strategy", errorbar="sd", ax=axes[1, 1])
axes[1, 1].set_title("Konsumsi Memori GPU Puncak (MB)", fontsize=13, fontweight='bold')

sns.stripplot(data=df_hasil, x='backbone', y='f1_macro', hue='strategy', dodge=True,
              color='black', size=3, legend=False, ax=axes[0, 0])  # titik = tiap seed
plt.tight_layout()
grafik_file = OUTPUT_DIR / f"grafik_hasil_evaluasi_{KODE}.png"
plt.savefig(grafik_file, dpi=300)
plt.show()
print(f"Grafik visualisasi tersimpan di: {grafik_file}")

## SEL 10 — Evaluasi Otomatis Hipotesis H1–H3 & Pertanyaan Riset

Sel berikut menguji H1–H3 secara kuantitatif dari data run. Hasilnya adalah *bukti*; narasi analisis tetap ditulis kelompok. Catatan: dengan n = 3 seed uji-t Welch bersifat indikatif (daya uji rendah), maka dilaporkan bersama selisih mean terhadap std.

In [ ]:
# ==============================================================================
# SEL 10: EVALUASI HIPOTESIS H1-H3 (DARI DATA, BUKAN ASUMSI)
# ==============================================================================
from scipy import stats

def _g(bb, st, kolom):
    return df_hasil[(df_hasil.backbone == bb) & (df_hasil.strategy == st)][kolom].values

R = {}
# H1: IndoBERT vs mBERT & DistilmBERT (full FT), F1 makro
for lawan in ("mbert", "distilmbert"):
    a, b = _g("indobert", "full", "f1_macro"), _g(lawan, "full", "f1_macro")
    t, p = stats.ttest_ind(a, b, equal_var=False)
    selisih = a.mean() - b.mean()
    R[f"H1 indobert vs {lawan}"] = (f"Δ={selisih*100:+.2f} poin, std=({a.std(ddof=1)*100:.2f}/{b.std(ddof=1)*100:.2f}), p={p:.4f}",
                                    bool(selisih > 0 and p < 0.05 and selisih > max(a.std(ddof=1), b.std(ddof=1))))

# H2: DistilmBERT vs mBERT (full FT)
m, d = "mbert", "distilmbert"
hemat_latih = 1 - _g(d, "full", "train_time_s").mean() / _g(m, "full", "train_time_s").mean()
hemat_lat = 1 - _g(d, "full", "latency_ms").mean() / _g(m, "full", "latency_ms").mean()
drop_f1 = (_g(m, "full", "f1_macro").mean() - _g(d, "full", "f1_macro").mean()) * 100
R["H2 hemat waktu latih >=35%"] = (f"{hemat_latih*100:.1f}%", hemat_latih >= 0.35)
R["H2 hemat latensi >=35%"] = (f"{hemat_lat*100:.1f}%", hemat_lat >= 0.35)
R["H2 penurunan F1 <=3 poin"] = (f"{drop_f1:.2f} poin", drop_f1 <= 3.0)

# H3: hierarki & efisiensi partial (per backbone)
for bb in CONFIG["backbones"]:
    f, p_, l = (_g(bb, s, "f1_macro").mean() for s in ("full", "partial", "probing"))
    R[f"H3 {bb} hierarki full>=partial>probing"] = (f"{f:.4f} / {p_:.4f} / {l:.4f}", bool(f >= p_ > l))
    R[f"H3 {bb} partial>=96% F1 full"] = (f"{p_/f*100:.1f}%", p_ / f >= 0.96)
    hw = 1 - _g(bb, "partial", "train_time_s").mean() / _g(bb, "full", "train_time_s").mean()
    hm = 1 - _g(bb, "partial", "peak_memory_mb").mean() / _g(bb, "full", "peak_memory_mb").mean()
    R[f"H3 {bb} hemat waktu >=40%"] = (f"{hw*100:.1f}%", hw >= 0.40)
    R[f"H3 {bb} hemat memori >=40%"] = (f"{hm*100:.1f}%", hm >= 0.40)

tabel_hipotesis = pd.DataFrame([(k, v[0], "TERDUKUNG" if v[1] else "TIDAK TERDUKUNG") for k, v in R.items()],
                               columns=["pengujian", "nilai terukur", "status"])
simpan_csv_atomik(tabel_hipotesis, OUTPUT_DIR / f"evaluasi_hipotesis_{KODE}.csv")
display(tabel_hipotesis)

### Jawaban 3 Pertanyaan Riset Wajib

> Teks resmi tiga pertanyaan tidak tercantum di PDF spesifikasi; salin dari panduan.md / arahan dosen, lalu jawab dengan merujuk **angka dari `rekap_agregat_smsa_<kode>.csv`** dan tabel hipotesis di atas.

**Pertanyaan 1:** _(salin)_ — Jawaban: ...

**Pertanyaan 2:** _(salin)_ — Jawaban: ...

**Pertanyaan 3:** _(salin)_ — Jawaban: ...

## Setelah SEL 9 Selesai

1. **Unduh artefak** dari `MyDrive/Studio3_NLP/output/` ke folder lokal `hasil_eksperimen/`
   (`hasil_smsa_<kode>.csv`, `rekap_agregat_smsa_<kode>.csv`, `grafik_hasil_evaluasi_<kode>.png`, `evaluasi_hipotesis_<kode>.csv`).
2. **Isi jawaban 3 Pertanyaan Riset** berdasarkan angka hasil run kelompok sendiri (Bab 8 panduan.md).
3. **Susun laporan singkat 3–4 halaman** (Bab 9) dan **slide maksimal 15** (Bab 10) ke folder
   `laporan_dan_presentasi/`.
4. **Export notebook (Run All) ke PDF** dengan nama
   `STUDIO3_Kelompok[X]_[NamaAnggota1]_[NamaAnggota2].pdf` (Bab 11 checklist).